# Supplement Sales Baseline Forecasts

This notebook compares four weekly forecasting baselines for each product: explicit ARIMA, Darts AutoARIMA, Prophet, and LightGBM. The train/test boundary is configured once, and all models use the same historical cutoff and future horizon.

In [1]:
import logging
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from darts import TimeSeries
from darts.models import AutoARIMA
from IPython.display import display
from lightgbm import LGBMRegressor
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.arima.model import ARIMA


warnings.filterwarnings("ignore")
logging.getLogger("prophet").setLevel(logging.ERROR)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

## 1. Configuration and data

In [2]:
DATA_PATH = Path("Supplement_Sales_Weekly_Expanded.csv")
DATE_COL = "Date"
PRODUCT_COL = "Product Name"
TARGET_CANDIDATES = ("Unit Sold", "Units Sold")
TRAIN_END_DATE = pd.Timestamp("2024-12-30")
PRODUCTS_TO_RUN = None  # None = train every product; e.g. ["Whey Protein"] = one product
PRODUCT_TO_PLOT = "Whey Protein"
ARIMA_ORDER = (2, 1, 2)
LAGS = (1, 2, 3, 4, 8, 12)
LIGHTGBM_CONTEXT_LAGS = (1, 2, 4, 8, 12)
LIGHTGBM_NUMERIC_CONTEXT = ("Price", "Revenue", "Discount", "Units Returned")
LIGHTGBM_STATIC_CONTEXT = ("Category",)
RANDOM_STATE = 42

if not DATA_PATH.exists():
    DATA_PATH = Path("implementations/supplement_sales/Supplement_Sales_Weekly_Expanded.csv")
sales = pd.read_csv(DATA_PATH, parse_dates=[DATE_COL])
target_col = next((column for column in TARGET_CANDIDATES if column in sales.columns), None)
if target_col is None:
    raise ValueError(f"Expected one of {TARGET_CANDIDATES}; found {sales.columns.tolist()}")

missing_context = [column for column in (*LIGHTGBM_NUMERIC_CONTEXT, *LIGHTGBM_STATIC_CONTEXT) if column not in sales.columns]
if missing_context:
    raise ValueError(f"Missing configured LightGBM context columns: {missing_context}")

sales = sales.sort_values([PRODUCT_COL, DATE_COL]).reset_index(drop=True)
sales[target_col] = pd.to_numeric(sales[target_col], errors="raise")
products = sales[PRODUCT_COL].dropna().unique().tolist()
if PRODUCTS_TO_RUN is None:
    products_to_run = products
else:
    unknown_products = sorted(set(PRODUCTS_TO_RUN) - set(products))
    if unknown_products:
        raise ValueError(f"Unknown product(s) in PRODUCTS_TO_RUN: {unknown_products}")
    products_to_run = list(dict.fromkeys(PRODUCTS_TO_RUN))
if PRODUCT_TO_PLOT not in products_to_run:
    raise ValueError("PRODUCT_TO_PLOT must be included in PRODUCTS_TO_RUN, or set PRODUCTS_TO_RUN=None.")

available_dates = pd.DatetimeIndex(sales[DATE_COL].drop_duplicates().sort_values())
if TRAIN_END_DATE not in available_dates:
    raise ValueError("TRAIN_END_DATE must be one of the weekly dates in the data.")

test_dates = available_dates[available_dates > TRAIN_END_DATE]
if len(test_dates) == 0:
    raise ValueError("TRAIN_END_DATE leaves no future observations for evaluation.")

print(f"Loaded {len(sales):,} rows, {len(products)} products, and {len(available_dates)} weekly dates.")
print(f"Training products: {len(products_to_run)}; target: {target_col}; train through {TRAIN_END_DATE.date()}; test horizon: {len(test_dates)} weeks.")
print(f"LightGBM context: numeric lags {LIGHTGBM_NUMERIC_CONTEXT}; static {LIGHTGBM_STATIC_CONTEXT}.")
display(sales.head())

Loaded 4,384 rows, 16 products, and 274 weekly dates.
Training products: 16; target: Units Sold; train through 2024-12-30; test horizon: 13 weeks.
LightGBM context: numeric lags ('Price', 'Revenue', 'Discount', 'Units Returned'); static ('Category',).


,Date,Product Name,Category,Units Sold,Price,Revenue,Discount,Units Returned,Location,Platform
0,2020-01-06,Ashwagandha,Herbal,181,15.490,"2,803.690",0.140,1,USA,Amazon
1,2020-01-13,Ashwagandha,Herbal,133,54.350,"7,228.550",0.160,1,UK,Walmart
2,2020-01-20,Ashwagandha,Herbal,156,23.490,"3,664.440",0.210,1,Canada,iHerb
3,2020-01-27,Ashwagandha,Herbal,174,30.940,"5,383.560",0.070,2,Canada,Amazon
4,2020-02-03,Ashwagandha,Herbal,135,19.210,"2,593.350",0.220,2,UK,Amazon


In [3]:
def product_frame(product_name):
    product = sales.loc[sales[PRODUCT_COL] == product_name].copy()
    product = product.set_index(DATE_COL).sort_index()
    product = product.reindex(available_dates)
    if product[target_col].isna().any():
        raise ValueError(f"{product_name} has missing weekly target values.")
    return product


def product_series(product_name):
    return product_frame(product_name)[target_col].astype(float)


def calendar_features(dates, trend_origin=None):
    dates = pd.DatetimeIndex(dates)
    trend_origin = dates.min() if trend_origin is None else pd.Timestamp(trend_origin)
    trend = (dates - trend_origin).days.to_numpy(dtype=float) / 7.0
    return pd.DataFrame({
        "year": dates.year,
        "month": dates.month,
        "quarter": dates.quarter,
        "week_of_year": dates.isocalendar().week.astype(int).to_numpy(),
        "trend": trend,
    }, index=dates)


def make_lagged_frame(series):
    frame = calendar_features(series.index, series.index.min())
    for lag in LAGS:
        frame[f"lag_{lag}"] = series.shift(lag)
    frame["target"] = series
    return frame.dropna()


def score_forecast(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    denominator = np.maximum((np.abs(actual) + np.abs(predicted)) / 2, 1e-8)
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "MAPE": np.mean(np.abs((actual - predicted) / np.maximum(np.abs(actual), 1e-8))) * 100,
        "sMAPE": np.mean(np.abs(actual - predicted) / denominator) * 100,
    }

## 2. Baseline model functions

ARIMA, AutoARIMA, Prophet, and Last Value use only the selected product's target history. LightGBM is the multivariate baseline: it uses target lags, calendar features, lagged numeric context (`Price`, `Revenue`, `Discount`, and `Units Returned`), and static `Category` features. Since future context values are not provided, LightGBM carries the last observed numeric context forward during recursive prediction. `Location` and `Platform` are excluded because they are not known for future weeks.

In [4]:
def forecast_last_value(train, horizon, context=None):
    return pd.Series(train.iloc[-1], index=test_dates, dtype=float)


def forecast_arima(train, horizon, context=None):
    model = ARIMA(train, order=ARIMA_ORDER, enforce_stationarity=False, enforce_invertibility=False)
    return pd.Series(model.fit().forecast(horizon), index=test_dates)


def forecast_auto_arima(train, horizon, context=None):
    series = TimeSeries.from_times_and_values(train.index, train.to_numpy(), freq="W-MON")
    model = AutoARIMA()
    model.fit(series)
    forecast = model.predict(n=horizon)
    return pd.Series(forecast.values().ravel(), index=test_dates)


def forecast_prophet(train, horizon, context=None):
    prophet_train = train.rename_axis("ds").rename("y").reset_index()
    model = Prophet(
        weekly_seasonality=False,
        daily_seasonality=False,
        yearly_seasonality=True,
        seasonality_mode="additive",
    )
    model.fit(prophet_train)
    future = model.make_future_dataframe(periods=horizon, freq="W-MON", include_history=False)
    forecast = model.predict(future)
    return pd.Series(forecast["yhat"].to_numpy(), index=test_dates)


def forecast_lightgbm(train, horizon, context):
    context = context.loc[train.index].copy()
    frame = calendar_features(train.index, train.index.min())
    for lag in LAGS:
        frame[f"target_lag_{lag}"] = train.shift(lag)
    for column in LIGHTGBM_NUMERIC_CONTEXT:
        numeric_context = pd.to_numeric(context[column], errors="raise")
        for lag in LIGHTGBM_CONTEXT_LAGS:
            frame[f"{column}_lag_{lag}"] = numeric_context.shift(lag)
    for column in LIGHTGBM_STATIC_CONTEXT:
        frame = pd.concat([frame, pd.get_dummies(context[column], prefix=column, dtype=float)], axis=1)
    frame["target"] = train
    frame = frame.dropna()
    feature_cols = [column for column in frame.columns if column != "target"]
    model = LGBMRegressor(
        objective="regression",
        n_estimators=250,
        learning_rate=0.03,
        num_leaves=15,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        verbosity=-1,
    )
    model.fit(frame[feature_cols], frame["target"])
    history = train.copy()
    context_history = context.copy()
    predictions = []
    for date in test_dates[:horizon]:
        row = calendar_features([date], train.index.min())
        for lag in LAGS:
            row[f"target_lag_{lag}"] = history.iloc[-lag]
        for column in LIGHTGBM_NUMERIC_CONTEXT:
            for lag in LIGHTGBM_CONTEXT_LAGS:
                row[f"{column}_lag_{lag}"] = context_history[column].iloc[-lag]
        for column in LIGHTGBM_STATIC_CONTEXT:
            row = pd.concat([row, pd.get_dummies(pd.Series([context_history[column].iloc[-1]]), prefix=column, dtype=float).set_index(row.index)], axis=1)
        row = row.reindex(columns=feature_cols, fill_value=0.0)
        prediction = float(model.predict(row)[0])
        predictions.append(prediction)
        history.loc[date] = prediction
        context_history.loc[date, list(LIGHTGBM_NUMERIC_CONTEXT)] = context_history.iloc[-1][list(LIGHTGBM_NUMERIC_CONTEXT)]
        context_history.loc[date, list(LIGHTGBM_STATIC_CONTEXT)] = context_history.iloc[-1][list(LIGHTGBM_STATIC_CONTEXT)]
    return pd.Series(predictions, index=test_dates[:horizon])

## 3. Fit all baselines and compare performance

In [5]:
model_functions = {
    "Last Value": forecast_last_value,
    "ARIMA": forecast_arima,
    "AutoARIMA": forecast_auto_arima,
    "Prophet": forecast_prophet,
    "LightGBM": forecast_lightgbm,
}

forecast_rows = []
failure_rows = []
forecast_store = {}
for product in products_to_run:
    product_data = product_frame(product)
    series = product_data[target_col].astype(float)
    train = series.loc[:TRAIN_END_DATE]
    context = product_data.loc[:TRAIN_END_DATE]
    actual = series.loc[test_dates]
    forecast_store[product] = {"actual": actual}
    for model_name, model_function in model_functions.items():
        try:
            predicted = model_function(train, len(test_dates), context).clip(lower=0)
            predicted = predicted.reindex(test_dates)
            forecast_store[product][model_name] = predicted
            forecast_rows.append({"Product Name": product, "Model": model_name, **score_forecast(actual, predicted)})
        except Exception as error:
            failure_rows.append({"Product Name": product, "Model": model_name, "Error": repr(error)})

metrics_by_product = pd.DataFrame(forecast_rows)
if failure_rows:
    display(pd.DataFrame(failure_rows))
    raise RuntimeError("At least one baseline failed; inspect the error table above.")

overall_comparison = (
    metrics_by_product.groupby("Model")[["MAE", "RMSE", "MAPE", "sMAPE"]]
    .mean()
    .sort_values("MAE")
)
display(overall_comparison.style.background_gradient(cmap="Blues", subset=["MAE", "RMSE", "MAPE", "sMAPE"]))

20:04:56 - cmdstanpy - INFO - Chain [1] start processing
20:04:56 - cmdstanpy - INFO - Chain [1] done processing
20:04:57 - cmdstanpy - INFO - Chain [1] start processing
20:04:57 - cmdstanpy - INFO - Chain [1] done processing
20:04:58 - cmdstanpy - INFO - Chain [1] start processing
20:04:58 - cmdstanpy - INFO - Chain [1] done processing
20:04:59 - cmdstanpy - INFO - Chain [1] start processing
20:04:59 - cmdstanpy - INFO - Chain [1] done processing
20:05:00 - cmdstanpy - INFO - Chain [1] start processing
20:05:00 - cmdstanpy - INFO - Chain [1] done processing
20:05:00 - cmdstanpy - INFO - Chain [1] start processing
20:05:00 - cmdstanpy - INFO - Chain [1] done processing
20:05:01 - cmdstanpy - INFO - Chain [1] start processing
20:05:01 - cmdstanpy - INFO - Chain [1] done processing
20:05:02 - cmdstanpy - INFO - Chain [1] start processing
20:05:02 - cmdstanpy - INFO - Chain [1] done processing
20:05:03 - cmdstanpy - INFO - Chain [1] start processing
20:05:03 - cmdstanpy - INFO - Chain [1]

,MAE,RMSE,MAPE,sMAPE
Model,,,,
AutoARIMA,10.122488,12.550394,6.835917,6.759403
Prophet,10.130641,12.524490,6.870568,6.766426
ARIMA,10.170643,12.510282,6.868971,6.791698
LightGBM,10.641316,13.273612,7.267520,7.100709
Last Value,15.697115,18.411903,10.596828,10.545352


In [6]:
product_comparison = metrics_by_product.pivot(index="Product Name", columns="Model", values="MAE").sort_index()
display(product_comparison.style.background_gradient(cmap="YlGn", axis=None))

fig = px.bar(
    overall_comparison.reset_index(),
    x="Model",
    y="MAE",
    title=f"Average test MAE by baseline | cutoff: {TRAIN_END_DATE.date()}",
    text_auto=".2f",
    color="Model",
    category_orders={"Model": list(model_functions)},
)
fig.update_layout(showlegend=False, yaxis_title="Mean absolute error (units sold)")
fig.show()

metric_heatmap = px.imshow(
    product_comparison,
    text_auto=".1f",
    aspect="auto",
    color_continuous_scale="YlGn",
    title="Product-level MAE by baseline (lower is better)",
    labels={"x": "Model", "y": "Product Name", "color": "MAE"},
)
metric_heatmap.show()

Model,ARIMA,AutoARIMA,Last Value,LightGBM,Prophet
Product Name,,,,,
Ashwagandha,14.073117,14.060713,14.000000,15.607615,15.545561
BCAA,6.913219,7.041561,9.000000,7.268007,6.794511
Biotin,8.698820,8.739667,10.000000,7.695189,9.271079
Collagen Peptides,6.834474,6.809800,10.461538,9.241588,7.108096
Creatine,8.671172,8.462357,10.384615,10.971237,10.425980
Electrolyte Powder,11.625909,11.652520,12.307692,11.757617,10.260788
Fish Oil,8.429399,8.878760,9.615385,9.110935,7.951227
Green Tea Extract,11.830260,11.562334,34.153846,13.853039,11.225429
Iron Supplement,10.999023,10.972591,15.000000,10.169253,10.091326


## 4. Inspect one product

Change `PRODUCT_TO_PLOT` in the configuration cell and re-run this cell to inspect another product.

In [7]:
if PRODUCT_TO_PLOT not in forecast_store:
    raise ValueError(f"Unknown product {PRODUCT_TO_PLOT!r}; choose from {products}")

series = product_series(PRODUCT_TO_PLOT)
plot_frame = pd.DataFrame({"Actual": series})
for model_name in model_functions:
    plot_frame[model_name] = forecast_store[PRODUCT_TO_PLOT][model_name]

fig = go.Figure()
fig.add_trace(go.Scatter(x=plot_frame.index, y=plot_frame["Actual"], name="Actual", mode="lines", line={"color": "#1f2937", "width": 2}))
colors = {"Last Value": "#6b7280", "ARIMA": "#2563eb", "AutoARIMA": "#dc2626", "Prophet": "#059669", "LightGBM": "#d97706"}
for model_name, color in colors.items():
    fig.add_trace(go.Scatter(x=test_dates, y=plot_frame.loc[test_dates, model_name], name=model_name, mode="lines+markers", line={"color": color, "dash": "dash"}))
fig.add_vline(x=TRAIN_END_DATE, line_dash="dot", line_color="#6b7280", annotation_text="train/test cutoff")
fig.update_layout(title=f"{PRODUCT_TO_PLOT}: actuals and baseline forecasts", xaxis_title="Date", yaxis_title=target_col, hovermode="x unified")
fig.show()

display(metrics_by_product.loc[metrics_by_product["Product Name"] == PRODUCT_TO_PLOT].sort_values("MAE"))

,Product Name,Model,MAE,RMSE,MAPE,sMAPE
73,Whey Protein,Prophet,8.184,10.070,5.234,5.261
72,Whey Protein,AutoARIMA,9.586,11.988,6.015,6.178
71,Whey Protein,ARIMA,9.705,11.838,6.130,6.257
74,Whey Protein,LightGBM,10.809,13.048,6.809,7.002
70,Whey Protein,Last Value,23.846,26.315,14.969,16.410



- The `Last Value` model is the naive floor required by the repository guidance. A more complex model should be judged against it, not only against the other fitted models.
- MAE, RMSE, MAPE, and sMAPE here are point-forecast diagnostics for this single chronological holdout. The repository's formal continuous-forecast leaderboard uses CRPS through the `ForecastingTask`/`backtest()` harness; add a rolling-origin `DataService` experiment before treating this table as a final model ranking.
- ARIMA, AutoARIMA, Prophet, and Last Value are univariate product-level baselines. LightGBM is multivariate, using lagged target/context features and static product category features.
- The LightGBM context forecast assumes that numeric context remains at its last observed value after the cutoff. This is a modeling assumption, not observed future information. If future prices, discounts, or returns are available from a planning system, replace the carried-forward values with those known future covariates.
- The product-level MAE heatmap and selected-product trajectory show where errors occur. For a full repository-style audit, extend the experiment with multiple origins, per-horizon CRPS, scored/skipped-origin counts, prediction intervals, and interval coverage.